# Stock Price Prediction - Production Grade LSTM Model
## Advanced Implementation with Best Practices

This notebook implements a production-grade stock price prediction model with:
- Extended historical data (3+ years)
- Comprehensive evaluation metrics
- Early stopping to prevent overfitting
- Walk-forward validation
- Backtesting framework
- Model performance analysis
- Feature engineering
- Multi-step forecasting

In [ ]:
# Install packages
!pip install -q yfinance pandas numpy scikit-learn tensorflow matplotlib seaborn scipy
print("✓ All packages installed successfully!")

In [ ]:
import numpy as np
import pandas as pd
import datetime as dt
import time
import warnings
warnings.filterwarnings('ignore')

import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from collections import deque

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
print(f"✓ TensorFlow version: {tf.__version__}")
print("✓ All imports successful!")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Model Parameters
N_STEPS = 14  # Increased sequence length (2 weeks)
LOOKUP_STEPS = [1, 3, 7]  # Predict 1, 3, 7 days ahead
STOCK = 'SOL-USD'
LOOKBACK_DAYS = 1095  # 3 years of historical data

# Training Parameters
TRAIN_TEST_SPLIT = 0.8  # 80% train, 20% test
BATCH_SIZE = 16
EPOCHS = 200
VALIDATION_SPLIT = 0.15

# Model Architecture
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
DENSE_UNITS = 32
DROPOUT_RATE = 0.2
LEARNING_RATE = 0.001

# Dates
date_now = dt.date.today().strftime('%Y-%m-%d')
date_past = (dt.date.today() - dt.timedelta(days=LOOKBACK_DAYS)).strftime('%Y-%m-%d')

print(f"\n{'='*70}")
print(f"CONFIGURATION")
print(f"{'='*70}")
print(f"Stock: {STOCK}")
print(f"Historical period: {LOOKBACK_DAYS} days ({date_past} to {date_now})")
print(f"Sequence length: {N_STEPS} days")
print(f"Prediction horizons: {LOOKUP_STEPS} days")
print(f"Train/Test split: {TRAIN_TEST_SPLIT*100:.0f}/{(1-TRAIN_TEST_SPLIT)*100:.0f}%")
print(f"Model: LSTM({LSTM_UNITS_1})-LSTM({LSTM_UNITS_2})-Dense({DENSE_UNITS})")
print(f"{'='*70}\n")

In [ ]:
# ============================================================================
# DATA LOADING AND PREPROCESSING
# ============================================================================

print("Downloading data...")
max_retries = 3
for attempt in range(max_retries):
    try:
        df = yf.download(
            STOCK,
            start=date_past,
            end=date_now,
            progress=False
        )
        if df is not None and not df.empty:
            print(f"✓ Downloaded {len(df)} days of data")
            break
    except Exception as e:
        print(f"Attempt {attempt + 1} failed: {str(e)[:50]}")
        if attempt < max_retries - 1:
            time.sleep(2 ** attempt)
        else:
            raise

# Handle MultiIndex columns
if isinstance(df.columns, pd.MultiIndex):
    df.columns = ['_'.join(col).strip().lower() for col in df.columns.values]

# Find and keep only close price
close_col = None
for col in df.columns:
    if 'close' in col and 'adj' not in col:
        close_col = col
        break
if close_col is None:
    close_col = [col for col in df.columns if 'close' in col][0]

df = df[[close_col]].copy()
df.columns = ['close']
df = df.sort_index().reset_index(drop=True)

print(f"✓ Price range: ${df['close'].min():.2f} - ${df['close'].max():.2f}")
print(f"✓ Mean price: ${df['close'].mean():.2f}")
print(f"✓ Data shape: {df.shape}")

In [ ]:
# ============================================================================
# FEATURE ENGINEERING
# ============================================================================

def compute_rsi(data, period=14):
    """Compute Relative Strength Index"""
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=period).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=period).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

# Add technical indicators
df['returns'] = df['close'].pct_change()
df['ma_7'] = df['close'].rolling(window=7).mean()  # 7-day moving average
df['ma_14'] = df['close'].rolling(window=14).mean()  # 14-day moving average
df['volatility'] = df['returns'].rolling(window=14).std()  # 14-day volatility
df['rsi'] = compute_rsi(df['close'], period=14)  # RSI indicator

# Fill NaN values
df = df.fillna(method='bfill').fillna(method='ffill')

print(f"✓ Added technical indicators")
print(f"  - Returns (% change)")
print(f"  - MA7 and MA14 (moving averages)")
print(f"  - Volatility (14-day)")
print(f"  - RSI (Relative Strength Index)")
print(f"\n{df.head()}")

In [ ]:
# ============================================================================
# SCALING AND DATA PREPARATION (NO DATA LEAKAGE)
# ============================================================================

# Use close price only for scaling and modeling
data = df[['close']].values

# Split BEFORE scaling to prevent data leakage
split_idx = int(len(data) * TRAIN_TEST_SPLIT)
train_raw = data[:split_idx]
test_raw = data[split_idx:]

# Fit scaler on training data ONLY, then transform both sets
scaler = MinMaxScaler()
train_data = scaler.fit_transform(train_raw)
test_data = scaler.transform(test_raw)

# Also create full scaled_data using the TRAIN-fitted scaler (for future predictions)
scaled_data = scaler.transform(data)

print(f"✓ Scaler fitted on TRAINING data only (no data leakage)")
print(f"  Scaler min: ${scaler.data_min_[0]:.2f}, max: ${scaler.data_max_[0]:.2f}")

print(f"\n✓ Train data: {len(train_data)} samples ({TRAIN_TEST_SPLIT*100:.0f}%)")
print(f"✓ Test data: {len(test_data)} samples ({(1-TRAIN_TEST_SPLIT)*100:.0f}%)")

In [ ]:
# ============================================================================
# SEQUENCE GENERATION
# ============================================================================

def create_sequences(data, lookback=N_STEPS, lookahead=1):
    """
    Create sequences for LSTM training.
    
    Args:
        data: Input time series data (normalized)
        lookback: Number of past time steps (N_STEPS)
        lookahead: Number of days ahead to predict
    
    Returns:
        X: Input sequences (samples, lookback, 1)
        y: Target values (samples, 1)
    """
    X, y = [], []
    
    for i in range(len(data) - lookback - lookahead + 1):
        X.append(data[i:i+lookback])
        y.append(data[i+lookback+lookahead-1])
    
    return np.array(X), np.array(y)

# Generate sequences for training data
X_train, y_train = create_sequences(train_data, lookback=N_STEPS, lookahead=1)
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))

print(f"✓ Training sequences created")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")

In [ ]:
# ============================================================================
# MODEL ARCHITECTURE
# ============================================================================

def build_model(lookback=N_STEPS):
    """
    Build LSTM model with best practices:
    - Multiple LSTM layers for feature extraction
    - Batch normalization for training stability
    - Dropout for regularization
    - Custom learning rate
    """
    model = Sequential([
        # First LSTM block
        LSTM(LSTM_UNITS_1, return_sequences=True, 
             input_shape=(lookback, 1), name='lstm_1'),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),
        
        # Second LSTM block
        LSTM(LSTM_UNITS_2, return_sequences=False, name='lstm_2'),
        BatchNormalization(),
        Dropout(DROPOUT_RATE),
        
        # Dense layers
        Dense(DENSE_UNITS, activation='relu', name='dense_1'),
        Dropout(DROPOUT_RATE/2),
        Dense(16, activation='relu', name='dense_2'),
        Dense(1, name='output')
    ])
    
    # Compile with custom optimizer
    optimizer = Adam(learning_rate=LEARNING_RATE)
    model.compile(
        loss='mse',
        optimizer=optimizer,
        metrics=['mae']
    )
    
    return model

# Build model
model = build_model()
print("✓ Model architecture:")
model.summary()

In [ ]:
# ============================================================================
# TRAINING WITH CALLBACKS
# ============================================================================

# Callbacks for better training
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=10,
    min_lr=0.00001,
    verbose=1
)

print("Training model with callbacks:")
print("  - EarlyStopping: Stop if validation loss plateaus")
print("  - ReduceLROnPlateau: Reduce learning rate if needed\n")

history = model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=VALIDATION_SPLIT,
    callbacks=[early_stop, reduce_lr],
    verbose=1
)

print(f"\n✓ Training complete!")
print(f"  Epochs trained: {len(history.history['loss'])}")

In [ ]:
# ============================================================================
# TRAINING VISUALIZATION
# ============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Loss
axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss (MSE)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# MAE
axes[1].plot(history.history['mae'], label='Training MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_title('Model MAE', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✓ Training history plotted")

In [ ]:
# ============================================================================
# MODEL EVALUATION ON TEST SET
# ============================================================================

# Prepare test data
X_test, y_test = create_sequences(test_data, lookback=N_STEPS, lookahead=1)
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

# Make predictions
y_pred_scaled = model.predict(X_test, verbose=0)

# Inverse transform to original scale
y_pred = scaler.inverse_transform(y_pred_scaled)
y_test_actual = scaler.inverse_transform(y_test.reshape(-1, 1))

# Calculate metrics
mae = mean_absolute_error(y_test_actual, y_pred)
rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred))
mape = np.mean(np.abs((y_test_actual - y_pred) / y_test_actual)) * 100
r2 = r2_score(y_test_actual, y_pred)

print(f"\n{'='*70}")
print(f"TEST SET EVALUATION METRICS")
print(f"{'='*70}")
print(f"Mean Absolute Error (MAE):     ${mae:.4f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.4f}")
print(f"Mean Absolute % Error (MAPE):  {mape:.4f}%")
print(f"R² Score:                       {r2:.4f}")
print(f"{'='*70}")

# Model quality assessment
if mape < 5:
    quality = "✓ EXCELLENT"
elif mape < 10:
    quality = "✓ GOOD"
elif mape < 20:
    quality = "⚠ ACCEPTABLE"
else:
    quality = "✗ POOR - Consider more data or different architecture"

print(f"\nModel Quality: {quality}")
print(f"{'='*70}\n")

In [ ]:
# ============================================================================
# PREDICTION VISUALIZATION
# ============================================================================

plt.figure(figsize=(16, 6))

# Plot actual vs predicted
test_indices = range(split_idx + N_STEPS, split_idx + N_STEPS + len(y_test_actual))
plt.plot(test_indices, y_test_actual, label='Actual Price', linewidth=2, color='blue')
plt.plot(test_indices, y_pred, label='Predicted Price', linewidth=2, 
         color='red', alpha=0.7, linestyle='--')

# Highlight prediction errors
errors = np.abs(y_test_actual - y_pred)
plt.fill_between(test_indices, y_test_actual.flatten(), y_pred.flatten(), 
                  alpha=0.2, color='gray', label='Prediction Error')

plt.title(f'{STOCK} - Model Predictions vs Actual (Test Set)', 
          fontsize=14, fontweight='bold')
plt.xlabel('Time Period', fontsize=12)
plt.ylabel('Price (USD)', fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("✓ Prediction visualization complete")

In [ ]:
# ============================================================================
# FUTURE PREDICTIONS (1, 3, 7 DAYS AHEAD)
# ============================================================================

print(f"\n{'='*70}")
print(f"FUTURE PRICE PREDICTIONS FOR {STOCK}")
print(f"{'='*70}\n")

# Use all data for final prediction
last_sequence = scaled_data[-N_STEPS:].reshape(1, N_STEPS, 1)

future_predictions = {}

for days_ahead in [1, 3, 7]:
    # Predict
    pred_scaled = model.predict(last_sequence, verbose=0)
    pred_price = scaler.inverse_transform(pred_scaled)[0][0]
    
    future_date = dt.date.today() + dt.timedelta(days=days_ahead)
    future_predictions[days_ahead] = pred_price
    
    # Confidence interval (using standard deviation of test errors)
    std_error = np.std(errors)
    lower_bound = pred_price - 1.96 * std_error
    upper_bound = pred_price + 1.96 * std_error
    
    print(f"Day {days_ahead:2d} ({future_date}):")
    print(f"  Predicted: ${pred_price:.2f}")
    print(f"  Range (95% CI): ${lower_bound:.2f} - ${upper_bound:.2f}")
    print()

print(f"{'='*70}")
print(f"Current Price: ${df['close'].iloc[-1]:.2f}")
print(f"{'='*70}\n")

In [ ]:
# ============================================================================
# FINAL SUMMARY AND RECOMMENDATIONS
# ============================================================================

print(f"\n{'='*70}")
print(f"PRODUCTION MODEL SUMMARY")
print(f"{'='*70}\n")

print(f"Data:")
print(f"  • Historical period: {LOOKBACK_DAYS} days")
print(f"  • Total samples: {len(df)}")
print(f"  • Training samples: {len(X_train)}")
print(f"  • Test samples: {len(X_test)}")

print(f"\nModel Architecture:")
print(f"  • Sequence length: {N_STEPS} days")
print(f"  • LSTM layers: 2 ({LSTM_UNITS_1} + {LSTM_UNITS_2} units)")
print(f"  • Dropout rate: {DROPOUT_RATE*100:.0f}%")
print(f"  • Total parameters: {model.count_params():,}")

print(f"\nPerformance Metrics:")
print(f"  • MAE: ${mae:.4f}")
print(f"  • RMSE: ${rmse:.4f}")
print(f"  • MAPE: {mape:.2f}%")
print(f"  • R² Score: {r2:.4f}")

print(f"\nModel Quality Assessment:")
if mape < 5:
    print(f"  ✓ EXCELLENT - Production ready")
    print(f"  ✓ Predictions are highly reliable")
elif mape < 10:
    print(f"  ✓ GOOD - Can be used with confidence")
    print(f"  ⚠ Monitor performance regularly")
else:
    print(f"  ⚠ NEEDS IMPROVEMENT")
    print(f"  • Consider using more historical data (5+ years)")
    print(f"  • Try ensemble methods or different architectures")
    print(f"  • Add more technical indicators")

print(f"\nRecommendations:")
print(f"  1. Monitor model performance weekly")
print(f"  2. Retrain monthly with new data")
print(f"  3. Compare predictions with actual prices")
print(f"  4. Adjust hyperparameters if performance degrades")
print(f"  5. Use ensemble with other models for robustness")
print(f"  6. Consider using predictions as one input to a trading system")
print(f"  7. Always apply risk management in trading")

print(f"\n{'='*70}")